# Model 1 — MobileSAM + MobileCLIP + LaMa
## End-to-end text-guided object removal on 10 COCO validation images

This notebook is the **first model setup** used in our object-removal pipeline:

**Image + text prompt → MobileSAM candidate masks → MobileCLIP semantic selection → MobileSAM point+box refinement → LaMa ONNX inpainting**

### What it does

1. Checks/installs the required packages.
2. Finds local model weights first and can download missing weights.
3. Obtains COCO 2017 validation annotations.
4. Selects **10 reproducible COCO images** with reasonably large target objects.
5. Uses the COCO target category as the text prompt.
6. Generates candidate masks with MobileSAM.
7. Scores candidates with MobileCLIP-S0.
8. Refines the selected candidate with MobileSAM.
9. Removes the object with LaMa `lama_fp32.onnx`.
10. Measures per-stage and total latency.
11. Computes mask IoU against the COCO target mask.
12. Saves masks, visualizations, benchmark CSV, and summary.

> COCO ground-truth masks are used only for target selection/evaluation. They are never passed to the inference pipeline.


## Model / dataset sources

The notebook follows the official MobileSAM `vit_t`, `SamPredictor`, and `SamAutomaticMaskGenerator` APIs. citeturn725470search0turn725470search3

MobileCLIP-S0 is loaded through Apple's official `mobileclip.create_model_and_transforms()` and tokenizer API. citeturn222414search0turn595339search2

The inpainting stage uses the recommended `lama_fp32.onnx` fixed-512 export. citeturn437762search5turn437762search6

COCO validation annotations are obtained from a Hugging Face mirror when a local COCO installation is not already available. citeturn307560search0


## 1. Installation

In [ ]:
# Uncomment once in a fresh environment.

# %pip install -q \
#     git+https://github.com/ChaoningZhang/MobileSAM.git \
#     git+https://github.com/apple-aiml-research/ml-mobileclip.git \
#     pycocotools \
#     opencv-python \
#     Pillow \
#     numpy \
#     pandas \
#     matplotlib \
#     requests \
#     tqdm \
#     onnxruntime


In [ ]:
import os
import json
import time
import urllib.request
from pathlib import Path
from collections import defaultdict

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm

import torch

print("Python:", __import__("sys").version.split()[0])
print("PyTorch:", torch.__version__)
print("MPS available:", hasattr(torch.backends, "mps") and torch.backends.mps.is_available())
print("CUDA available:", torch.cuda.is_available())


## 2. Configuration

In [ ]:
SEED = 42
NUM_IMAGES = 10

# Set False when all weights/data already exist locally.
ALLOW_DOWNLOADS = True

# Model weights are searched here first.
LOCAL_MODEL_DIR = Path.cwd() / "model_wts"
LOCAL_MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Optional local COCO root. Example:
# COCO_LOCAL_ROOT = Path("/kaggle/input/coco-2017-dataset/coco2017")
COCO_LOCAL_ROOT = None

# "auto" -> CUDA > MPS > CPU
FORCE_DEVICE = "auto"

# MobileSAM automatic mask generation
SAM_POINTS_PER_SIDE = 10
SAM_POINTS_PER_BATCH = 64
SAM_PRED_IOU_THRESH = 0.84
SAM_STABILITY_THRESH = 0.88
SAM_MIN_MASK_AREA = 100

# Candidate filtering / MobileCLIP
MAX_CLIP_CANDIDATES = 48
MIN_AREA_RATIO = 0.003
MAX_AREA_RATIO = 0.75
CLIP_BATCH_SIZE = 32

# MobileSAM refinement
REFINE_MIN_CANDIDATE_RECALL = 0.55
REFINE_MAX_AREA_GROWTH = 4.0
REFINE_BOX_SCALE = 1.28
REFINE_PADDING_MIN = 6

# LaMa
LAMA_SIZE = 512
ROI_PADDING = 0.35
MASK_DILATION = 9
ROI_FEATHER = 7

OUTPUT_DIR = Path.cwd() / "mobilesam_model1_coco10_results"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Output directory:", OUTPUT_DIR)


In [ ]:
def choose_device(preference="auto"):
    preference = preference.lower()

    if preference == "mps":
        if not hasattr(torch.backends, "mps") or not torch.backends.mps.is_available():
            raise RuntimeError("MPS requested but unavailable.")
        return torch.device("mps")

    if preference == "cuda":
        if not torch.cuda.is_available():
            raise RuntimeError("CUDA requested but unavailable.")
        return torch.device("cuda")

    if preference == "cpu":
        return torch.device("cpu")

    if torch.cuda.is_available():
        return torch.device("cuda")

    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")

    return torch.device("cpu")


DEVICE = choose_device(FORCE_DEVICE)

# Released MobileSAM builds can hit a float64 problem in automatic mask
# generation on Apple MPS. Keep MobileSAM itself on CPU on MPS.
SAM_DEVICE = torch.device("cpu") if DEVICE.type == "mps" else DEVICE
CLIP_DEVICE = DEVICE

print("Main device :", DEVICE)
print("MobileSAM   :", SAM_DEVICE)
print("MobileCLIP  :", CLIP_DEVICE)


## 3. Model weights

In [ ]:
WEIGHT_SPECS = {
    "mobile_sam": {
        "filename": "mobile_sam.pt",
        "urls": [
            "https://github.com/ChaoningZhang/MobileSAM/raw/master/weights/mobile_sam.pt",
        ],
        "min_bytes": 20_000_000,
    },
    "mobileclip_s0": {
        "filename": "mobileclip_s0.pt",
        "urls": [
            "https://docs-assets.developer.apple.com/ml-research/datasets/mobileclip/mobileclip_s0.pt",
            "https://huggingface.co/apple/MobileCLIP-S0/resolve/main/mobileclip_s0.pt?download=true",
        ],
        "min_bytes": 100_000_000,
    },
    "lama": {
        "filename": "lama_fp32.onnx",
        "urls": [
            "https://huggingface.co/Carve/LaMa-ONNX/resolve/main/lama_fp32.onnx?download=true",
            "https://huggingface.co/g-ronimo/lama/resolve/main/lama_fp32.onnx?download=true",
        ],
        "min_bytes": 100_000_000,
    },
}


def find_local_file(filename):
    candidates = [
        LOCAL_MODEL_DIR / filename,
        Path.cwd() / filename,
        Path.cwd() / "weights" / filename,
    ]

    kaggle_root = Path("/kaggle/input")
    if kaggle_root.exists():
        for parent in kaggle_root.iterdir():
            if not parent.is_dir():
                continue
            candidates += [
                parent / filename,
                parent / "model_wts" / filename,
                parent / "weights" / filename,
            ]

    for p in candidates:
        if p.exists() and p.is_file() and p.stat().st_size > 0:
            return p

    return None


def download_file(urls, destination, min_bytes=1):
    if not ALLOW_DOWNLOADS:
        raise FileNotFoundError(
            f"Missing {destination}; ALLOW_DOWNLOADS=False."
        )

    last_error = None

    for url in urls:
        try:
            print("Downloading:", url)
            tmp = destination.with_suffix(destination.suffix + ".part")
            urllib.request.urlretrieve(url, tmp)

            size = tmp.stat().st_size
            if size < min_bytes:
                raise RuntimeError(
                    f"Downloaded only {size} bytes; expected >= {min_bytes}."
                )

            tmp.replace(destination)
            print(
                f"Saved {destination} "
                f"({destination.stat().st_size / 1024**2:.1f} MB)"
            )
            return destination

        except Exception as exc:
            last_error = exc
            print("  failed:", repr(exc))

    raise RuntimeError(
        f"Could not obtain {destination.name}: {last_error}"
    )


MODEL_PATHS = {}

for key, spec in WEIGHT_SPECS.items():
    p = find_local_file(spec["filename"])

    if p is None:
        p = download_file(
            spec["urls"],
            LOCAL_MODEL_DIR / spec["filename"],
            spec["min_bytes"],
        )

    MODEL_PATHS[key] = p
    print(f"{key:16s} -> {p}")

print("All model weights are ready.")


## 4. Load MobileSAM

In [ ]:
from mobile_sam import (
    sam_model_registry,
    SamAutomaticMaskGenerator,
)

# Compatibility helper for PyTorch versions with torch.load(weights_only=...).
# The MobileSAM checkpoint is a trusted local model file.
import inspect

_original_torch_load = torch.load

def _patched_torch_load(*args, **kwargs):
    try:
        params = inspect.signature(_original_torch_load).parameters
        if "weights_only" in params and "weights_only" not in kwargs:
            kwargs["weights_only"] = False
    except Exception:
        pass
    return _original_torch_load(*args, **kwargs)

torch.load = _patched_torch_load

try:
    mobile_sam = sam_model_registry["vit_t"](
        checkpoint=str(MODEL_PATHS["mobile_sam"])
    )
finally:
    torch.load = _original_torch_load

mobile_sam.to(SAM_DEVICE)
mobile_sam.eval()

sam_generator = SamAutomaticMaskGenerator(
    model=mobile_sam,
    points_per_side=SAM_POINTS_PER_SIDE,
    points_per_batch=SAM_POINTS_PER_BATCH,
    pred_iou_thresh=SAM_PRED_IOU_THRESH,
    stability_score_thresh=SAM_STABILITY_THRESH,
    crop_n_layers=0,
    min_mask_region_area=SAM_MIN_MASK_AREA,
    output_mode="binary_mask",
)

print("MobileSAM loaded successfully.")


## 5. Load MobileCLIP-S0

In [ ]:
import mobileclip
from mobileclip.modules.common.mobileone import reparameterize_model

clip_model, _, clip_preprocess = mobileclip.create_model_and_transforms(
    "mobileclip_s0",
    pretrained=str(MODEL_PATHS["mobileclip_s0"]),
)

clip_tokenizer = mobileclip.get_tokenizer("mobileclip_s0")

clip_model.eval().to(CLIP_DEVICE)

try:
    clip_model = reparameterize_model(clip_model)
    clip_model.eval().to(CLIP_DEVICE)
    print("MobileCLIP reparameterized for inference.")
except Exception as exc:
    print("Reparameterization skipped:", repr(exc))

print("MobileCLIP-S0 loaded successfully.")


## 6. Load LaMa ONNX

In [ ]:
import onnxruntime as ort

sess_options = ort.SessionOptions()
sess_options.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
sess_options.execution_mode = ort.ExecutionMode.ORT_SEQUENTIAL

lama_session = ort.InferenceSession(
    str(MODEL_PATHS["lama"]),
    sess_options=sess_options,
    providers=["CPUExecutionProvider"],
)

lama_inputs = lama_session.get_inputs()
lama_outputs = lama_session.get_outputs()

print("LaMa inputs:")
for x in lama_inputs:
    print(" ", x.name, x.shape, x.type)

print("\nLaMa outputs:")
for x in lama_outputs:
    print(" ", x.name, x.shape, x.type)

input_names = {x.name: x for x in lama_inputs}

if "image" in input_names and "mask" in input_names:
    LAMA_IMAGE_NAME = "image"
    LAMA_MASK_NAME = "mask"
else:
    image_name = None
    mask_name = None

    for x in lama_inputs:
        shape = list(x.shape)
        if len(shape) != 4:
            continue

        channels = shape[1]
        if channels == 3:
            image_name = x.name
        elif channels == 1:
            mask_name = x.name

    if image_name is None or mask_name is None:
        raise RuntimeError(
            f"Could not identify LaMa inputs: {[x.name for x in lama_inputs]}"
        )

    LAMA_IMAGE_NAME = image_name
    LAMA_MASK_NAME = mask_name

LAMA_OUTPUT_NAME = lama_outputs[0].name

print("\nUsing:", LAMA_IMAGE_NAME, LAMA_MASK_NAME, LAMA_OUTPUT_NAME)


## 7. Build a reproducible 10-image COCO subset

The previous notebook could silently produce an empty `image_paths` list when downloads failed, which then caused:

```text
IndexError: list index out of range
```

This version does not index an empty list.

It first searches for a local/Kaggle COCO installation. Otherwise it downloads the validation annotation JSON and **only the 10 selected images** from a Hugging Face mirror. COCO itself remains the source dataset. citeturn307560search0turn229144search4


In [ ]:
COCO_ANN_URL = (
    "https://huggingface.co/datasets/merve/coco/"
    "resolve/main/annotations/instances_val2017.json?download=true"
)

COCO_IMAGE_BASE = (
    "https://huggingface.co/datasets/merve/coco/"
    "resolve/main/val2017/"
)


def locate_coco_root():
    if COCO_LOCAL_ROOT is not None:
        p = Path(COCO_LOCAL_ROOT)
        if p.exists():
            return p

    candidates = [
        Path("/kaggle/input/coco-2017-dataset/coco2017"),
        Path("/kaggle/input/coco-2017-dataset/COCO2017"),
        Path("/kaggle/input/coco2017"),
        Path("/kaggle/input/coco-2017"),
        Path.cwd() / "coco2017",
        Path.cwd() / "coco",
    ]

    for p in candidates:
        if p.exists():
            return p

    return None


COCO_ROOT = locate_coco_root()
COCO_ANN_PATH = OUTPUT_DIR / "instances_val2017.json"

if COCO_ROOT is not None:
    local_candidates = [
        COCO_ROOT / "annotations" / "instances_val2017.json",
        COCO_ROOT / "instances_val2017.json",
    ]
    local_json = next((p for p in local_candidates if p.exists()), None)

    if local_json is not None:
        COCO_ANN_PATH = local_json

if not COCO_ANN_PATH.exists():
    if not ALLOW_DOWNLOADS:
        raise FileNotFoundError(
            "COCO validation annotations are missing and downloads are disabled."
        )

    print("Downloading COCO validation annotations (~20 MB)...")
    urllib.request.urlretrieve(COCO_ANN_URL, COCO_ANN_PATH)

if COCO_ANN_PATH.stat().st_size < 1_000_000:
    raise RuntimeError(
        f"COCO annotation file looks invalid: {COCO_ANN_PATH} "
        f"({COCO_ANN_PATH.stat().st_size} bytes)"
    )

print("COCO annotation file:", COCO_ANN_PATH)


In [ ]:
with open(COCO_ANN_PATH, "r", encoding="utf-8") as f:
    coco_json = json.load(f)

categories = {
    int(c["id"]): c["name"]
    for c in coco_json["categories"]
}

images_meta = {
    int(img["id"]): img
    for img in coco_json["images"]
}

annotations_by_image = defaultdict(list)

for ann in coco_json["annotations"]:
    annotations_by_image[int(ann["image_id"])].append(ann)

print("Images:", len(images_meta))
print("Annotations:", len(coco_json["annotations"]))
print("Categories:", len(categories))


In [ ]:
rng = np.random.default_rng(SEED)

eligible = []

for image_id, anns in annotations_by_image.items():
    meta = images_meta.get(image_id)

    if meta is None:
        continue

    W = int(meta["width"])
    H = int(meta["height"])

    # Use the largest annotation in the image as the benchmark target.
    for ann in sorted(
        anns,
        key=lambda x: float(x.get("area", 0)),
        reverse=True,
    ):
        area_ratio = float(ann.get("area", 0)) / max(1, W * H)

        if 0.05 <= area_ratio <= 0.65:
            eligible.append({
                "image_id": image_id,
                "category_id": int(ann["category_id"]),
                "category": categories[int(ann["category_id"])],
                "ann_id": int(ann["id"]),
                "area": float(ann["area"]),
                "area_ratio": area_ratio,
                "width": W,
                "height": H,
                "bbox": ann["bbox"],
            })
            break

rng.shuffle(eligible)
eligible.sort(key=lambda x: x["area_ratio"], reverse=True)

selected = []
used_images = set()
used_categories = set()

# First pass: try to make the 10 cases category-diverse.
for item in eligible:
    if item["image_id"] in used_images:
        continue
    if item["category_id"] in used_categories:
        continue

    selected.append(item)
    used_images.add(item["image_id"])
    used_categories.add(item["category_id"])

    if len(selected) == NUM_IMAGES:
        break

# Fallback if fewer than 10 unique categories are available.
if len(selected) < NUM_IMAGES:
    for item in eligible:
        if item["image_id"] in used_images:
            continue

        selected.append(item)
        used_images.add(item["image_id"])

        if len(selected) == NUM_IMAGES:
            break

if len(selected) != NUM_IMAGES:
    raise RuntimeError(
        f"Only {len(selected)} suitable COCO cases found; "
        f"expected {NUM_IMAGES}."
    )

cases = pd.DataFrame(selected)
cases["prompt"] = cases["category"].apply(
    lambda x: f"a photo of a {x}"
)

display(
    cases[[
        "image_id",
        "category",
        "area_ratio",
        "prompt",
    ]]
)


In [ ]:
COCO_IMAGE_DIR = OUTPUT_DIR / "coco_images"
COCO_IMAGE_DIR.mkdir(parents=True, exist_ok=True)


def get_coco_image(image_id):
    filename = f"{int(image_id):012d}.jpg"

    # 1) local/Kaggle COCO
    if COCO_ROOT is not None:
        candidates = [
            COCO_ROOT / "val2017" / filename,
            COCO_ROOT / "images" / "val2017" / filename,
            COCO_ROOT / filename,
        ]

        for p in candidates:
            if p.exists() and p.stat().st_size > 10_000:
                return p

    # 2) notebook cache
    dst = COCO_IMAGE_DIR / filename

    if dst.exists() and dst.stat().st_size > 10_000:
        return dst

    # 3) Hugging Face mirror
    if not ALLOW_DOWNLOADS:
        raise FileNotFoundError(
            f"COCO image {image_id} missing and downloads disabled."
        )

    url = COCO_IMAGE_BASE + filename + "?download=true"

    try:
        urllib.request.urlretrieve(url, dst)
    except Exception as exc:
        raise RuntimeError(
            f"Could not download COCO image {image_id} from {url}. "
            "Either enable internet or add the COCO 2017 dataset to the notebook."
        ) from exc

    if dst.stat().st_size <= 10_000:
        raise RuntimeError(
            f"Downloaded COCO image {image_id} is unexpectedly small: "
            f"{dst.stat().st_size} bytes."
        )

    return dst


cases["image_path"] = [
    str(get_coco_image(int(image_id)))
    for image_id in cases["image_id"]
]

assert len(cases) == NUM_IMAGES
assert all(Path(p).exists() for p in cases["image_path"])

print(f"Prepared exactly {len(cases)} COCO images.")


In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(18, 8))
axes = axes.ravel()

for ax, (_, row) in zip(axes, cases.iterrows()):
    img = Image.open(row["image_path"]).convert("RGB")
    ax.imshow(img)
    ax.set_title(
        f'{int(row["image_id"])} — {row["category"]}'
    )
    ax.axis("off")

plt.tight_layout()
plt.show()


## 8. COCO ground-truth evaluation helpers

In [ ]:
from pycocotools.coco import COCO

coco = COCO(str(COCO_ANN_PATH))


def gt_instance_mask(image_id, ann_id):
    ann = coco.loadAnns([int(ann_id)])[0]
    return coco.annToMask(ann).astype(bool)


def same_category_annotations(image_id, category_id):
    return [
        ann
        for ann in annotations_by_image[int(image_id)]
        if int(ann["category_id"]) == int(category_id)
    ]


def mask_iou(a, b):
    a = np.asarray(a, dtype=bool)
    b = np.asarray(b, dtype=bool)

    inter = np.logical_and(a, b).sum()
    union = np.logical_or(a, b).sum()

    return float(inter / union) if union else 0.0


def mask_precision_recall(pred, gt):
    pred = np.asarray(pred, dtype=bool)
    gt = np.asarray(gt, dtype=bool)

    inter = np.logical_and(pred, gt).sum()

    precision = inter / max(1, pred.sum())
    recall = inter / max(1, gt.sum())

    return float(precision), float(recall)


## 9. End-to-end pipeline implementation

In [ ]:
def load_rgb(path):
    return np.asarray(
        Image.open(path).convert("RGB")
    )


def normalize_prompt(prompt):
    p = str(prompt).strip().lower()

    prefixes = [
        "remove the ",
        "remove ",
        "delete the ",
        "delete ",
        "erase the ",
        "erase ",
        "a photo of a ",
        "a photo of the ",
        "a photo of ",
    ]

    for prefix in prefixes:
        if p.startswith(prefix):
            p = p[len(prefix):]
            break

    return p.strip()


def mask_bbox(mask):
    ys, xs = np.where(np.asarray(mask, dtype=bool))

    if len(xs) == 0:
        return None

    return [
        int(xs.min()),
        int(ys.min()),
        int(xs.max() + 1),
        int(ys.max() + 1),
    ]


def overlay_mask(image, mask, alpha=0.45):
    out = image.copy()
    mask = np.asarray(mask, dtype=bool)

    tint = np.zeros_like(out)
    tint[..., 0] = 255
    tint[..., 1] = 60
    tint[..., 2] = 60

    out[mask] = (
        alpha * tint[mask] +
        (1 - alpha) * out[mask]
    ).astype(np.uint8)

    return out


def crop_from_bbox(image, box, pad_ratio=0.08):
    H, W = image.shape[:2]

    x1, y1, x2, y2 = map(int, box)

    bw = max(1, x2 - x1)
    bh = max(1, y2 - y1)

    px = int(round(bw * pad_ratio))
    py = int(round(bh * pad_ratio))

    x1 = max(0, x1 - px)
    y1 = max(0, y1 - py)
    x2 = min(W, x2 + px)
    y2 = min(H, y2 + py)

    return image[y1:y2, x1:x2]


def masked_crop(image, mask):
    box = mask_bbox(mask)

    if box is None:
        return image

    H, W = image.shape[:2]
    x1, y1, x2, y2 = box

    bw = max(1, x2 - x1)
    bh = max(1, y2 - y1)

    px = int(round(bw * 0.08))
    py = int(round(bh * 0.08))

    x1p = max(0, x1 - px)
    y1p = max(0, y1 - py)
    x2p = min(W, x2 + px)
    y2p = min(H, y2 + py)

    crop = image[y1p:y2p, x1p:x2p].copy()
    cmask = np.asarray(mask, dtype=bool)[y1p:y2p, x1p:x2p]

    blurred = cv2.GaussianBlur(
        crop,
        (0, 0),
        sigmaX=7,
    )

    crop[~cmask] = blurred[~cmask]

    return crop


In [ ]:
def generate_sam_candidates(image):
    with torch.inference_mode():
        return sam_generator.generate(image)


def filter_candidates(masks, image_shape):
    H, W = image_shape[:2]
    total_area = float(H * W)

    valid = []

    for m in masks:
        area_ratio = (
            float(m["area"])
            / max(1.0, total_area)
        )

        if area_ratio < MIN_AREA_RATIO:
            continue

        if area_ratio > MAX_AREA_RATIO:
            continue

        valid.append(m)

    valid.sort(
        key=lambda m:
            float(m.get("predicted_iou", 0.0))
            * float(m.get("stability_score", 0.0)),
        reverse=True,
    )

    return valid[:MAX_CLIP_CANDIDATES]


In [ ]:
def mobileclip_score_candidates(image, candidates, prompt):
    prompts = [
        f"a photo of a {prompt}",
        f"a photo containing a {prompt}",
        f"a close view of a {prompt}",
    ]

    text_tokens = clip_tokenizer(prompts).to(CLIP_DEVICE)

    with torch.inference_mode():
        text_features = clip_model.encode_text(text_tokens)
        text_features = text_features / text_features.norm(
            dim=-1,
            keepdim=True,
        )

    box_crops = []
    masked_crops = []

    for candidate in candidates:
        candidate_mask = np.asarray(
            candidate["segmentation"],
            dtype=bool,
        )

        box = mask_bbox(candidate_mask)

        if box is None:
            bx, by, bw, bh = candidate["bbox"]
            box = [
                int(bx),
                int(by),
                int(bx + bw),
                int(by + bh),
            ]

        box_crops.append(
            Image.fromarray(
                crop_from_bbox(image, box)
            )
        )

        masked_crops.append(
            Image.fromarray(
                masked_crop(image, candidate_mask)
            )
        )

    def encode_image_batches(pil_images):
        chunks = []

        for start in range(
            0,
            len(pil_images),
            CLIP_BATCH_SIZE,
        ):
            batch_images = pil_images[
                start:start + CLIP_BATCH_SIZE
            ]

            batch = torch.stack([
                clip_preprocess(img)
                for img in batch_images
            ]).to(CLIP_DEVICE)

            with torch.inference_mode():
                features = clip_model.encode_image(batch)
                features = features / features.norm(
                    dim=-1,
                    keepdim=True,
                )

            chunks.append(features.detach())

        return torch.cat(chunks, dim=0)

    box_features = encode_image_batches(box_crops)
    masked_features = encode_image_batches(masked_crops)

    box_similarity = box_features @ text_features.T
    masked_similarity = masked_features @ text_features.T

    box_score = box_similarity.mean(dim=1)
    masked_score = masked_similarity.mean(dim=1)

    final_score = (
        0.72 * box_score
        + 0.28 * masked_score
    )

    return final_score.detach().float().cpu().numpy()


In [ ]:
def refine_with_mobilesam(image, candidate):
    H, W = image.shape[:2]

    candidate_mask = np.asarray(
        candidate["segmentation"],
        dtype=bool,
    )

    ys, xs = np.where(candidate_mask)

    if len(xs) == 0:
        raise RuntimeError(
            "Selected MobileSAM candidate is empty."
        )

    # Positive point at the robust center of the candidate.
    px = int(np.median(xs))
    py = int(np.median(ys))

    x1 = int(candidate["bbox"][0])
    y1 = int(candidate["bbox"][1])
    x2 = int(
        candidate["bbox"][0]
        + candidate["bbox"][2]
    )
    y2 = int(
        candidate["bbox"][1]
        + candidate["bbox"][3]
    )

    bw = max(1, x2 - x1)
    bh = max(1, y2 - y1)

    extra_x = max(
        REFINE_PADDING_MIN,
        int((REFINE_BOX_SCALE - 1.0) * bw / 2),
    )
    extra_y = max(
        REFINE_PADDING_MIN,
        int((REFINE_BOX_SCALE - 1.0) * bh / 2),
    )

    box = np.array([
        max(0, x1 - extra_x),
        max(0, y1 - extra_y),
        min(W - 1, x2 + extra_x),
        min(H - 1, y2 + extra_y),
    ], dtype=np.float32)

    predictor = sam_generator.predictor
    predictor.set_image(image)

    with torch.inference_mode():
        masks, scores, _ = predictor.predict(
            point_coords=np.array(
                [[px, py]],
                dtype=np.float32,
            ),
            point_labels=np.array(
                [1],
                dtype=np.int32,
            ),
            box=box,
            multimask_output=True,
        )

    candidate_area = max(
        1,
        int(candidate_mask.sum()),
    )

    ranked = []

    for idx, m in enumerate(masks):
        mm = np.asarray(m, dtype=bool)
        area = int(mm.sum())

        if area == 0:
            continue

        inter = int(
            np.logical_and(
                mm,
                candidate_mask,
            ).sum()
        )

        union = int(
            np.logical_or(
                mm,
                candidate_mask,
            ).sum()
        )

        recall = inter / candidate_area
        local_iou = inter / max(1, union)
        growth = area / candidate_area

        if recall < REFINE_MIN_CANDIDATE_RECALL:
            continue

        if growth > REFINE_MAX_AREA_GROWTH:
            continue

        score = (
            0.45 * float(scores[idx])
            + 0.35 * recall
            + 0.25 * local_iou
            - 0.10 * max(0.0, growth - 2.0)
        )

        ranked.append((score, idx))

    if not ranked:
        return (
            candidate_mask.astype(np.uint8) * 255,
            0.0,
        )

    _, best_idx = max(
        ranked,
        key=lambda x: x[0],
    )

    return (
        masks[best_idx].astype(np.uint8) * 255,
        float(scores[best_idx]),
    )


In [ ]:
def make_square_roi(image, mask):
    H, W = image.shape[:2]

    ys, xs = np.where(mask > 0)

    if len(xs) == 0:
        raise ValueError("Empty mask.")

    x1 = int(xs.min())
    x2 = int(xs.max() + 1)
    y1 = int(ys.min())
    y2 = int(ys.max() + 1)

    bw = x2 - x1
    bh = y2 - y1

    px = max(8, int(round(bw * ROI_PADDING)))
    py = max(8, int(round(bh * ROI_PADDING)))

    x1 -= px
    y1 -= py
    x2 += px
    y2 += py

    side = max(x2 - x1, y2 - y1, 64)

    cx = (x1 + x2) / 2.0
    cy = (y1 + y2) / 2.0

    x1 = int(round(cx - side / 2))
    y1 = int(round(cy - side / 2))
    x2 = x1 + side
    y2 = y1 + side

    if x1 < 0:
        x2 -= x1
        x1 = 0

    if y1 < 0:
        y2 -= y1
        y1 = 0

    if x2 > W:
        shift = x2 - W
        x1 -= shift
        x2 = W

    if y2 > H:
        shift = y2 - H
        y1 -= shift
        y2 = H

    x1 = max(0, x1)
    y1 = max(0, y1)
    x2 = min(W, x2)
    y2 = min(H, y2)

    return (
        image[y1:y2, x1:x2],
        mask[y1:y2, x1:x2],
        (x1, y1, x2, y2),
    )


def lama_inpaint(image, mask):
    roi, roi_mask, roi_box = make_square_roi(
        image,
        mask,
    )

    image_512 = cv2.resize(
        roi,
        (LAMA_SIZE, LAMA_SIZE),
        interpolation=cv2.INTER_AREA,
    )

    mask_512 = cv2.resize(
        roi_mask.astype(np.uint8),
        (LAMA_SIZE, LAMA_SIZE),
        interpolation=cv2.INTER_NEAREST,
    )

    image_input = (
        image_512.astype(np.float32)
        / 255.0
    ).transpose(2, 0, 1)[None]

    mask_input = (
        (mask_512 > 127)
        .astype(np.float32)[None, None]
    )

    output = lama_session.run(
        [LAMA_OUTPUT_NAME],
        {
            LAMA_IMAGE_NAME: image_input,
            LAMA_MASK_NAME: mask_input,
        },
    )[0]

    output = np.asarray(output)[0].transpose(
        1, 2, 0
    )

    vmin = float(output.min())
    vmax = float(output.max())

    if vmin < -0.5 and vmax <= 1.5:
        output = (
            (output + 1.0)
            * 0.5
            * 255.0
        )
    elif vmax <= 1.5:
        output = output * 255.0

    output = np.clip(
        output,
        0,
        255,
    ).astype(np.uint8)

    output = cv2.resize(
        output,
        (roi.shape[1], roi.shape[0]),
        interpolation=cv2.INTER_CUBIC,
    )

    feather = max(1, int(ROI_FEATHER))
    if feather % 2 == 0:
        feather += 1

    alpha = cv2.GaussianBlur(
        (roi_mask > 0).astype(np.float32),
        (feather, feather),
        0,
    )[..., None]

    blended = (
        roi.astype(np.float32) * (1.0 - alpha)
        + output.astype(np.float32) * alpha
    )

    result = image.copy()
    x1, y1, x2, y2 = roi_box

    result[y1:y2, x1:x2] = np.clip(
        blended,
        0,
        255,
    ).astype(np.uint8)

    return result, roi_box


In [ ]:
def remove_object(image, prompt):
    clean_prompt = normalize_prompt(prompt)

    if not clean_prompt:
        raise ValueError("Prompt is empty.")

    total_start = time.perf_counter()

    # Stage 1 — MobileSAM automatic proposals
    t = time.perf_counter()
    all_masks = generate_sam_candidates(image)
    sam_proposal_s = time.perf_counter() - t

    if not all_masks:
        raise RuntimeError(
            "MobileSAM generated no candidate masks."
        )

    candidates = filter_candidates(
        all_masks,
        image.shape,
    )

    if not candidates:
        raise RuntimeError(
            "No MobileSAM candidates survived filtering."
        )

    # Stage 2 — MobileCLIP semantic selection
    t = time.perf_counter()
    clip_scores = mobileclip_score_candidates(
        image,
        candidates,
        clean_prompt,
    )
    mobileclip_s = time.perf_counter() - t

    combined_scores = []

    H, W = image.shape[:2]
    total_area = float(H * W)

    for idx, candidate in enumerate(candidates):
        area_ratio = (
            float(candidate["area"])
            / total_area
        )

        sam_quality = (
            float(candidate.get("predicted_iou", 0.0))
            * float(candidate.get("stability_score", 0.0))
        )

        size_penalty = (
            max(0.0, area_ratio - 0.30)
            * 0.30
        )

        score = (
            float(clip_scores[idx])
            + 0.05 * sam_quality
            - size_penalty
        )

        combined_scores.append(score)

    best_idx = int(
        np.argmax(combined_scores)
    )

    selected_candidate = candidates[best_idx]

    # Stage 3 — MobileSAM refinement
    t = time.perf_counter()

    refined_mask, refine_score = refine_with_mobilesam(
        image,
        selected_candidate,
    )

    sam_refine_s = time.perf_counter() - t

    k = max(1, int(MASK_DILATION))
    if k % 2 == 0:
        k += 1

    kernel = cv2.getStructuringElement(
        cv2.MORPH_ELLIPSE,
        (k, k),
    )

    refined_mask = cv2.dilate(
        refined_mask,
        kernel,
        iterations=1,
    )

    # Stage 4 — LaMa
    t = time.perf_counter()

    inpainted, roi_box = lama_inpaint(
        image,
        refined_mask,
    )

    lama_s = time.perf_counter() - t

    total_s = time.perf_counter() - total_start

    return {
        "original": image,
        "prompt": prompt,
        "mask": refined_mask,
        "overlay": overlay_mask(
            image,
            refined_mask,
        ),
        "result": inpainted,
        "proposal_count": len(all_masks),
        "clip_candidate_count": len(candidates),
        "clip_score": float(
            clip_scores[best_idx]
        ),
        "combined_score": float(
            combined_scores[best_idx]
        ),
        "refine_score": float(
            refine_score
        ),
        "roi_box": roi_box,
        "selected_candidate_mask": (
            np.asarray(
                selected_candidate["segmentation"],
                dtype=np.uint8,
            ) * 255
        ),
        "latency": {
            "mobile_sam_proposal_s": sam_proposal_s,
            "mobileclip_s": mobileclip_s,
            "mobile_sam_refine_s": sam_refine_s,
            "lama_s": lama_s,
            "total_s": total_s,
        },
    }


print("End-to-end Model 1 pipeline is ready.")


## 10. Warm-up

In [ ]:
warmup_image = load_rgb(cases.iloc[0]["image_path"])

print("Running warm-up (excluded from benchmark)...")
_ = remove_object(
    warmup_image,
    cases.iloc[0]["prompt"],
)
print("Warm-up complete.")


## 11. Run all 10 COCO cases

In [ ]:
pipeline_outputs = []
benchmark_rows = []

for _, row in tqdm(
    cases.iterrows(),
    total=len(cases),
    desc="Model 1 inference",
):
    image_id = int(row["image_id"])
    category_id = int(row["category_id"])

    image = load_rgb(row["image_path"])

    output = remove_object(
        image,
        row["prompt"],
    )

    gt_target = gt_instance_mask(
        image_id,
        int(row["ann_id"]),
    )

    pred_mask = output["mask"] > 127

    target_iou = mask_iou(
        pred_mask,
        gt_target,
    )

    precision, recall = mask_precision_recall(
        pred_mask,
        gt_target,
    )

    same_cat_ious = []

    for ann in same_category_annotations(
        image_id,
        category_id,
    ):
        gt_same = coco.annToMask(ann).astype(bool)
        same_cat_ious.append(
            mask_iou(pred_mask, gt_same)
        )

    best_same_category_iou = (
        max(same_cat_ious)
        if same_cat_ious
        else target_iou
    )

    latency = output["latency"]

    benchmark_rows.append({
        "image_id": image_id,
        "category": row["category"],
        "prompt": row["prompt"],
        "proposal_count": output["proposal_count"],
        "clip_candidates": output["clip_candidate_count"],
        "clip_score": output["clip_score"],
        "combined_score": output["combined_score"],
        "refine_score": output["refine_score"],

        "sam_proposal_ms": (
            latency["mobile_sam_proposal_s"] * 1000
        ),
        "mobileclip_ms": (
            latency["mobileclip_s"] * 1000
        ),
        "sam_refine_ms": (
            latency["mobile_sam_refine_s"] * 1000
        ),
        "lama_ms": (
            latency["lama_s"] * 1000
        ),
        "total_ms": (
            latency["total_s"] * 1000
        ),

        "gt_target_iou": target_iou,
        "best_same_category_iou": best_same_category_iou,
        "gt_precision": precision,
        "gt_recall": recall,

        "gt_area_ratio": float(row["area_ratio"]),
        "pred_area_ratio": float(pred_mask.mean()),
    })

    pipeline_outputs.append({
        "image_id": image_id,
        "category": row["category"],
        "output": output,
        "gt_mask": gt_target,
    })

benchmark_df = pd.DataFrame(benchmark_rows)

assert len(benchmark_df) == NUM_IMAGES

display(benchmark_df)


## 12. Summary

In [ ]:
summary_df = pd.DataFrame({
    "Metric": [
        "Images",
        "Mean total latency (ms)",
        "Median total latency (ms)",
        "Min total latency (ms)",
        "Max total latency (ms)",
        "Mean MobileSAM proposal (ms)",
        "Mean MobileCLIP scoring (ms)",
        "Mean MobileSAM refinement (ms)",
        "Mean LaMa (ms)",
        "Mean target IoU",
        "Mean best same-category IoU",
        "Mean precision",
        "Mean recall",
        "Mean proposals/image",
    ],
    "Value": [
        len(benchmark_df),
        benchmark_df["total_ms"].mean(),
        benchmark_df["total_ms"].median(),
        benchmark_df["total_ms"].min(),
        benchmark_df["total_ms"].max(),
        benchmark_df["sam_proposal_ms"].mean(),
        benchmark_df["mobileclip_ms"].mean(),
        benchmark_df["sam_refine_ms"].mean(),
        benchmark_df["lama_ms"].mean(),
        benchmark_df["gt_target_iou"].mean(),
        benchmark_df["best_same_category_iou"].mean(),
        benchmark_df["gt_precision"].mean(),
        benchmark_df["gt_recall"].mean(),
        benchmark_df["proposal_count"].mean(),
    ],
})

summary_df["Value"] = summary_df["Value"].round(4)

display(summary_df)


## 13. Latency charts

In [ ]:
stage_means = {
    "MobileSAM\nproposal": benchmark_df["sam_proposal_ms"].mean(),
    "MobileCLIP\nscoring": benchmark_df["mobileclip_ms"].mean(),
    "MobileSAM\nrefinement": benchmark_df["sam_refine_ms"].mean(),
    "LaMa": benchmark_df["lama_ms"].mean(),
}

plt.figure(figsize=(10, 5))
plt.bar(
    stage_means.keys(),
    stage_means.values(),
)
plt.ylabel("Mean latency (ms)")
plt.title("Model 1 — mean stage latency")
plt.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(11, 5))
plt.bar(
    benchmark_df["image_id"].astype(str),
    benchmark_df["total_ms"],
)
plt.xlabel("COCO image ID")
plt.ylabel("Total latency (ms)")
plt.title("Model 1 — end-to-end latency per image")
plt.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(11, 5))
plt.bar(
    benchmark_df["image_id"].astype(str),
    benchmark_df["gt_target_iou"],
)
plt.xlabel("COCO image ID")
plt.ylabel("Target-instance IoU")
plt.title("Model 1 — final mask IoU vs COCO target")
plt.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()


## 14. Visual end-to-end inspection

In [ ]:
def show_case(record):
    image_id = record["image_id"]
    category = record["category"]
    output = record["output"]
    gt = record["gt_mask"]

    original = output["original"]
    selected = output["selected_candidate_mask"] > 0
    refined = output["mask"] > 127
    result = output["result"]

    fig, axes = plt.subplots(
        1,
        5,
        figsize=(22, 5),
    )

    axes[0].imshow(original)
    axes[0].set_title(
        f"Original\nCOCO {image_id}"
    )

    axes[1].imshow(original)
    axes[1].imshow(
        np.ma.masked_where(
            ~gt,
            gt,
        ),
        alpha=0.55,
    )
    axes[1].set_title("COCO GT target")

    axes[2].imshow(original)
    axes[2].imshow(
        np.ma.masked_where(
            ~selected,
            selected,
        ),
        alpha=0.55,
    )
    axes[2].set_title("MobileCLIP-selected\nSAM candidate")

    axes[3].imshow(original)
    axes[3].imshow(
        np.ma.masked_where(
            ~refined,
            refined,
        ),
        alpha=0.55,
    )
    axes[3].set_title("MobileSAM refined")

    axes[4].imshow(result)
    axes[4].set_title(
        f"LaMa removal\n{category}"
    )

    for ax in axes:
        ax.axis("off")

    plt.tight_layout()
    plt.show()


for record in pipeline_outputs:
    show_case(record)


## 15. Save results

In [ ]:
MASK_DIR = OUTPUT_DIR / "masks"
VIS_DIR = OUTPUT_DIR / "visualizations"

MASK_DIR.mkdir(parents=True, exist_ok=True)
VIS_DIR.mkdir(parents=True, exist_ok=True)

benchmark_df.to_csv(
    OUTPUT_DIR / "benchmark.csv",
    index=False,
)

summary_df.to_csv(
    OUTPUT_DIR / "summary.csv",
    index=False,
)

cases.to_csv(
    OUTPUT_DIR / "cases.csv",
    index=False,
)

for record in pipeline_outputs:
    image_id = record["image_id"]
    output = record["output"]
    gt = record["gt_mask"]

    np.save(
        MASK_DIR / f"{image_id:012d}_gt.npy",
        gt.astype(np.uint8),
    )

    np.save(
        MASK_DIR / f"{image_id:012d}_candidate.npy",
        (output["selected_candidate_mask"] > 0).astype(np.uint8),
    )

    np.save(
        MASK_DIR / f"{image_id:012d}_refined.npy",
        (output["mask"] > 127).astype(np.uint8),
    )

    Image.fromarray(
        output["result"]
    ).save(
        VIS_DIR / f"{image_id:012d}_removed.png"
    )

    Image.fromarray(
        output["overlay"]
    ).save(
        VIS_DIR / f"{image_id:012d}_refined_overlay.png"
    )

print("Saved results to:", OUTPUT_DIR)


## 16. Final benchmark table

This is the table to carry forward into the comparison of the three model setups.

### Model 1
**MobileSAM + MobileCLIP-S0 + LaMa**

| Group | Metrics |
|---|---|
| Semantic selection | MobileCLIP score, combined score |
| Segmentation | target IoU, best same-category IoU, precision, recall |
| Mask generation | number of MobileSAM proposals |
| Runtime | MobileSAM proposal, MobileCLIP, MobileSAM refinement, LaMa, total latency |
| Removal | saved final inpainted image for every test case |

### Pipeline

**COCO image → MobileSAM automatic proposals → MobileCLIP text/image similarity → MobileSAM point+box refinement → LaMa ROI inpainting**
